# 第11章：策略框架

## 本章学习目标

- 理解策略设计模式
- 掌握交易决策机制
- 实现信号策略
- 能够开发自定义策略

---

## 11.1 策略框架概述

Qlib 的策略框架将模型预测转换为具体的交易决策。

### 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                    策略框架流程                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   模型预测  ──→  策略  ──→  交易决策  ──→  执行器          │
│   (Score)       (Signal)   (Decision)    (Executor)         │
│                                                             │
│   特点:                                                      │
│   - 模块化设计                                              │
│   - 支持多种策略类型                                        │
│   - 灵活的权重分配                                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 策略类型

| 策略类型 | 说明 | 代表类 |
|----------|------|--------|
| 信号策略 | 基于预测信号选股 | SignalStrategy |
| Topk策略 | 选择评分最高的k只股票 | TopkStrategy |
| 权重策略 | 分配股票权重 | WeightStrategyBase |

In [ ]:
import qlib
from qlib.strategy.base import BaseStrategy
from qlib.contrib.strategy.signal_strategy import TopkStrategy, SoftTopkStrategy
from qlib.backtest.decision import TradeDecisionWO, Order
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 11.2 策略基类

In [ ]:
# 查看 BaseStrategy 基类
from qlib.strategy.base import BaseStrategy
import inspect

print("BaseStrategy 主要方法:")
print("=" * 50)

for name, method in inspect.getmembers(BaseStrategy, predicate=inspect.isfunction):
    if not name.startswith("_"):
        print(f"  - {name}")

In [ ]:
# 关键方法说明
print("策略关键接口:")
print("=" * 60)

methods = [
    ("generate_trade_decision()", "生成交易决策，核心方法"),
    ("reset_level()", "重置策略级别"),
    ("get_risk_degree()", "获取风险敞口"),
]

for method, desc in methods:
    print(f"  {method:30s} - {desc}")

## 11.3 信号策略 (SignalStrategy)

In [ ]:
# 创建模拟信号
# 假设我们有一些股票的评分

# 获取沪深300股票列表
from qlib.data import D

instruments = D.instruments(market="csi300")
stock_list = instruments['instrument'].tolist()[:50]  # 取前50只演示

# 创建模拟信号
np.random.seed(42)
dates = pd.date_range("2022-01-01", "2022-03-31", freq="B")

signal_data = {}
for stock in stock_list:
    signal_data[stock] = np.random.randn(len(dates))

signal = pd.DataFrame(signal_data, index=dates)
signal.index.name = "datetime"

# 转换为长格式
signal_long = signal.stack()
signal_long.index.names = ['datetime', 'instrument']
signal_long.name = 'score'

print(f"信号数据形状: {signal_long.shape}")
signal_long.head()

In [ ]:
# 使用 TopkStrategy
# 该策略选择评分最高的 topk 只股票

print("TopkStrategy 参数说明:")
print("=" * 50)
print("  signal: 预测信号 (Series，带 instrument 和 datetime 索引)")
print("  topk: 选择的股票数量")
print("  n_drop: 每期卖出上期不在topk中的股票数量")

In [ ]:
# 创建 TopkStrategy 实例
from qlib.contrib.strategy.signal_strategy import TopkStrategy

# 注意：实际使用需要在回测环境中
# 这里仅演示策略创建

topk_strategy = TopkStrategy(
    signal=signal_long,
    topk=30,
    n_drop=5,
)

print(f"TopkStrategy 创建成功")
print(f"  topk: {topk_strategy.topk}")
print(f"  n_drop: {topk_strategy.n_drop}")

## 11.4 SoftTopkStrategy

In [ ]:
# SoftTopkStrategy: 使用 softmax 分配权重
from qlib.contrib.strategy.signal_strategy import SoftTopkStrategy

print("SoftTopkStrategy 参数说明:")
print("=" * 50)
print("  signal: 预测信号")
print("  topk: 选择的股票数量")
print("  separate: 是否分离买入和卖出")
print("  temperature: Softmax 温度参数")

In [ ]:
# 创建 SoftTopkStrategy 实例
soft_topk_strategy = SoftTopkStrategy(
    signal=signal_long,
    topk=30,
    separate=False,
)

print(f"SoftTopkStrategy 创建成功")

## 11.5 自定义策略

In [ ]:
# 实现自定义策略
from qlib.strategy.base import BaseStrategy
from qlib.backtest.decision import TradeDecisionWO, Order

class MyMomentumStrategy(BaseStrategy):
    """简单的动量策略示例"""
    
    def __init__(self, signal, topk=30, risk_degree=0.95):
        """
        参数:
            signal: 预测信号 Series
            topk: 持仓股票数量
            risk_degree: 仓位比例
        """
        self.signal = signal
        self.topk = topk
        self.risk_degree = risk_degree
        super().__init__()
    
    def generate_trade_decision(self, execute_result=None):
        """生成交易决策"""
        # 获取当前交易日
        trade_step = self.trade_calendar.get_trade_step()
        
        # 获取当日信号
        # 注意：实际实现需要考虑交易日历和信号对齐
        
        # 创建订单列表
        order_list = []
        
        # ... 具体的交易逻辑
        
        return TradeDecisionWO(order_list, self)

print("MyMomentumStrategy 定义完成")

## 11.6 权重策略

In [ ]:
# 权重分配策略
from qlib.contrib.strategy.order_generator import OrderGenerator

print("权重分配方法:")
print("=" * 50)

weight_methods = {
    "等权重": "每只股票分配相同权重",
    "市值加权": "按市值比例分配权重",
    "信号加权": "按信号强度分配权重",
    "风险平价": "按风险贡献分配权重",
}

for method, desc in weight_methods.items():
    print(f"\n{method}:")
    print(f"  {desc}")

In [ ]:
# 演示权重计算
def equal_weight(n_assets):
    """等权重"""
    return np.ones(n_assets) / n_assets

def signal_weight(signals):
    """信号加权（正信号比例分配）"""
    positive_mask = signals > 0
    if positive_mask.sum() == 0:
        return equal_weight(len(signals))
    
    weights = np.zeros(len(signals))
    weights[positive_mask] = signals[positive_mask]
    weights = weights / weights.sum()
    return weights

# 示例
example_signals = np.array([0.5, 0.3, -0.1, 0.4, 0.2])

print("示例信号:", example_signals)
print("\n等权重:", equal_weight(len(example_signals)))
print("信号加权:", signal_weight(example_signals))

## 11.7 交易决策对象

In [ ]:
# 交易决策对象说明
from qlib.backtest.decision import TradeDecisionWO, TradeDecision

print("交易决策类型:")
print("=" * 50)

decision_types = {
    "TradeDecisionWO": "仅包含订单的决策 (Without Position)",
    "TradeDecision": "包含订单和持仓的决策",
}

for name, desc in decision_types.items():
    print(f"\n{name}:")
    print(f"  {desc}")

In [ ]:
# Order 对象说明
from qlib.backtest.decision import Order

print("Order 对象属性:")
print("=" * 50)

order_attrs = [
    ("stock_id", "股票代码"),
    ("amount", "交易数量（正数买入，负数卖出）"),
    ("direction", "交易方向 (1=买入, -1=卖出)"),
    ("type", "订单类型 (市场单/限价单)"),
]

for attr, desc in order_attrs:
    print(f"  {attr:15s} - {desc}")

## 11.8 策略信号可视化

In [ ]:
import matplotlib.pyplot as plt

# 可视化信号分布
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 信号分布
axes[0, 0].hist(signal_long.values, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('信号分布')
axes[0, 0].set_xlabel('信号值')
axes[0, 0].set_ylabel('频数')

# 2. 每日信号统计
daily_stats = signal.unstack().agg(['mean', 'std'], axis=1)
axes[0, 1].plot(daily_stats.index, daily_stats['mean'], label='均值')
axes[0, 1].fill_between(daily_stats.index, 
                        daily_stats['mean'] - daily_stats['std'],
                        daily_stats['mean'] + daily_stats['std'],
                        alpha=0.3, label='±1σ')
axes[0, 1].set_title('每日信号统计')
axes[0, 1].set_xlabel('日期')
axes[0, 1].set_ylabel('信号值')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Topk 选择示例
sample_date = dates[10]
sample_signal = signal.loc[sample_date].sort_values(ascending=False)

axes[1, 0].bar(range(len(sample_signal)), sample_signal.values, color='steelblue')
axes[1, 0].axvline(x=30-0.5, color='red', linestyle='--', label=f'topk=30')
axes[1, 0].set_title(f'{sample_date.strftime("%Y-%m-%d")} 信号排名')
axes[1, 0].set_xlabel('排名')
axes[1, 0].set_ylabel('信号值')
axes[1, 0].legend()

# 4. 热力图
import seaborn as sns
signal_matrix = signal.iloc[:20, :20]  # 取部分数据
sns.heatmap(signal_matrix.T, cmap='RdBu_r', center=0, ax=axes[1, 1],
            xticklabels=False, yticklabels=False)
axes[1, 1].set_title('信号热力图 (部分)')

plt.tight_layout()
plt.show()

## 11.9 实践练习

In [ ]:
# 练习1: 实现一个基于波动率的策略
# 当波动率超过阈值时减仓

# 你的代码



# class VolatilityStrategy(BaseStrategy):
#     def __init__(self, signal, volatility_threshold=0.03):
#         ...
#     
#     def generate_trade_decision(self, execute_result=None):
#         ...

In [ ]:
# 练习2: 实现一个行业中性策略
# 在每个行业内选择 topk 股票

# 你的代码



# class SectorNeutralStrategy(BaseStrategy):
#     def __init__(self, signal, sector_mapping, topk_per_sector=5):
#         ...

In [ ]:
# 练习3: 分析不同 topk 值对策略的影响
# 对比 topk=10, 20, 30, 50 的效果

# 你的代码



# for topk in [10, 20, 30, 50]:
#     strategy = TopkStrategy(signal=signal_long, topk=topk)
#     # 模拟选股结果
#     print(f"topk={topk}")

## 11.10 本章小结

本章我们学习了：

1. **策略框架结构**：
   - BaseStrategy 基类
   - generate_trade_decision() 核心方法

2. **内置策略**：
   - TopkStrategy: 选择评分最高的 k 只股票
   - SoftTopkStrategy: 使用 softmax 分配权重

3. **自定义策略**：
   - 继承 BaseStrategy
   - 实现交易逻辑

4. **交易决策对象**：
   - TradeDecisionWO
   - Order

### 关键 API 速查

```python
# 创建 TopkStrategy
strategy = TopkStrategy(
    signal=predictions,  # 预测信号
    topk=30,             # 持仓数量
)

# 自定义策略
class MyStrategy(BaseStrategy):
    def generate_trade_decision(self, execute_result=None):
        # 实现交易逻辑
        return TradeDecisionWO(order_list, self)
```

### 下一章预告

下一章我们将学习回测系统，包括：
- 回测引擎架构
- 账户和持仓管理
- 交易成本分析